# Python pour biochimistes: Integrer des données provenant d'une ressource externe dans un objet BioPython

## Introduction

Ce n'est malheureusement pas toujours aussi facile... Certaines bases de données, comme Ensembl, ne peuvent être accéder directement via BioPython. Mais ça ne veut pas dire que nous ne pouvons pas se servir de BioPython.

Dans l'exemple qui va suivre, on va accéder à Ensembl via l'interface REST (https://rest.ensembl.org) pour extraire des données que nous convertirons en éléments utilisables pour BioPython pour créer des objets `Seq` et `SeqRecord`.

Dans [l'exemple précédent](python_oo_demo_biopython_1_intro), la classe `SeqIO` retourne des objets `SeqRecord`, une version plus musclée des objets `Seq`. Pourquoi faire plus? Un objet `Seq` est une représentation minimale de l'information disponible pour une séquence biologique: la séquence elle-même et pas beaucoup plus. On a souvent, sinon toujours, beaucoup plus d'informations sur la séquence, correspondant aux méta-données: nom de la séquence, description, exons, transcrits et ainsi de suite.

Si nous n'avons pas accès à une méthode dans la classe `SeqIO` pour lire et interpréter directement les infos, on doit travailler une peu plus... Dans le code qui suivra, nous utiliserons le service web REST d'Ensembl pour aller chercher des éléments d'info et nous peuplerons un objet SeqRecord, étape par étape.

## Étape 1 - Création de l'objet SeqRecord avec infos minimales ##

À la base, un objet `SeqRecord` contient les infos suivantes:

 - Un objet `Seq` avec une chaine de caractères, nucléotides ou acides aminés;
 - un champ avec un identificateur, idéalement unique;
 - un nom;
 - une description

Prenons toujours notre gène préféré: la cortactine chez H. sapiens, symbol `CTTN` (évidemment, ce ne sont pas les vrais infos :-) ): 

In [2]:
#
# Importons toutes les librairies nécessaires
#
import requests
import json
import pprint
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
#
# On ne peut créer un objet SeqRecord minimal
# simplement en créant une instance de l'objet...
#
# Les items Seq et id sont obligatoirements définis
# pour une création réussie
#
aSeq = SeqRecord(
    Seq("N"),
    id="CTTN"
)
print(aSeq.id)
print(aSeq.seq)

CTTN
N


## Étape 2 - Augmentons les infos en utilisant les données ENSEMBL ##

Mais la création de l'objet `SeqRecord` amène aussi la création implicite de champs qui restent vides jusqu'à temps que l'on y mette les infos désirées ;-) Lorsqu'on utilise les méthodes de la classe `SeqIO` pour lire des fichiers en format FASTA ou Genbank, elles remplissent ces champs à partir des infos contenues dans le fichier. Mais comment faire si on a des données pour lesquelles `SeqIO` n'a pas de méthodes?  

Commençons pas mettre les bonnes infos concernant CTTN en utilisant le service [rest.ensembl.org](https://rest.ensembl.org). Pour utiliser ce service efficacement, on doit jeter un coupe d'oeil sur la [doc de construction des URL](https://rest.ensembl.org) capables de nous donné ce que l'on cherche. Mais pour l'exemple, on va tricher une peu ;-) On va commencer directement avec la fonction `lookup` en spécifiant que nous cherchons CTTN par son symbole dans les infos sur Homo sapiens.  

In [18]:
#
# Importons les librairies nécessaires
#
import requests
import json
import pprint
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, SimpleLocation

#
# Évidemment, c'est le serveur à utiliser
#
server = "https://rest.ensembl.org/"
#
# On cherche les infos pour un seul gène: CTTN
# pour H sapiens
#
#query = "lookup/symbol/homo_sapiens/CTTN?expand=1"
query = "lookup/symbol/homo_sapiens/CTTN"
#
# Rappel: la méthode get se construit ainsi:
# requests.get(URL,params=un_dict,headers=un_dict)
# mais comme l'URL pour le service sépare les champs avec '/',
# params est pas nécessaire...
#
# On veut recevoir les infos en mode JSON
#
reponse = requests.get(server + query, headers={ "Content-Type" : "application/json"})
#
# Si on reçoit un signal positif (<400)
#
if reponse.ok:
    data = reponse.json()
    #
    # Ça permet d'afficher à l'écran de manière plus
    # lisible
    #
    pprint.pprint(data)

{'assembly_name': 'GRCh38',
 'biotype': 'protein_coding',
 'canonical_transcript': 'ENST00000301843.13',
 'db_type': 'core',
 'description': 'cortactin [Source:HGNC Symbol;Acc:HGNC:3338]',
 'display_name': 'CTTN',
 'end': 70436584,
 'id': 'ENSG00000085733',
 'logic_name': 'ensembl_havana_gene_homo_sapiens',
 'object_type': 'Gene',
 'seq_region_name': '11',
 'source': 'ensembl_havana',
 'species': 'homo_sapiens',
 'start': 70397704,
 'strand': 1,
 'version': 18}


Ok, on voit que l'on obtient un dictionnaire avec des informations fort utiles. On a déjà la description réelle mais aussi d'autres infos que nous pourrons utiliser avec d'autres recherches avec le service REST, particulièrement le champs `id` (qui n'est pas équivalent au champs `id` de l'objet `SeqRecord`) qui est l'identificateur unique du gène pour faire des recherches ultérieures.

Commençons par ajouter la description: 

In [20]:
#
# Ajoutons une valeur pour le champs description 
# de l'objet aSeq, à partir de data
#
aSeq.description = data["description"]
#
# Ajoutons une valeur pour le champs name 
# de l'objet aSeq, à partir de data
#
aSeq.name = data["display_name"]
#
# Vérifions notre changement
#
print(aSeq.name+" : "+aSeq.description)

CTTN : cortactin [Source:HGNC Symbol;Acc:HGNC:3338]


## Étape 3 - Structure des données à l'intérieur d'un objet SeqRecord et comment s'y prendre pour utiliser les données externes ##

Comme expliquer ci-dessus, la création d'un objet `SeqRecord` entraine la création de structures de données qui au début sont vides:

- `.name`: une variable de type `string` pour désigner l'objet;
- `.description`: une variable de type `string` contenant des infos complémentaires sur l'objet;
- `.annnotations`: une variable de type `dict`, contenant des données spécifiques à l'objet lui-même et non pas à des portions de l'objet
- `.features`: un objet `SeqFeature`, contenant des données spécifiques à une portion de l'objet (par exemple: un exon pour un transcrit donné)
- `.letter_annotations`: une variable de type `list` ou `tuples`, de même longueur que la séquence elle-même, et contenant une valeur par lettre (par exemple: les valeurs Q d'une séquence FASTQ ou bien des valeurs d'une analyse de structure secondaire dans une séquence de protéine)
- `.dbxrefs`: une variable de type `list` contenant des infos textuelles sur une référence croisée; par exemple: "NCBI Gene:2017"

Avec notre première recherche, on a déjà des infos à mettre dans le champs `.annotations`:

In [21]:
#
# Chaque SeqRecord devrait avoir un nom unique; pourquoi ne pas
# utiliser l'identificateur unique du gène chez Ensembl
#
aSeq.id = data["id"]

aSeq.annotations["species"] = data["species"]
aSeq.annotations["assembly_name"] = data["assembly_name"]
aSeq.annotations["symbol"] = data["display_name"]
aSeq.annotations["biotype"] = data["biotype"]
aSeq.annotations["canonical_transcript"] = data["canonical_transcript"]
# Pas obliger de garder le nom de la clé d'origine ;-)
aSeq.annotations["chromosome"] = data["seq_region_name"]
aSeq.annotations["strand"] = data["strand"]
aSeq.annotations["start"] = data["start"]
aSeq.annotations["end"] = data["end"]
#
# Il faut aller chercher l'itérateur de aSeq
#
for cle, valeur in aSeq.annotations.items():
    print(cle+":"+str(valeur))

species:homo_sapiens
assembly_name:GRCh38
symbol:CTTN
biotype:protein_coding
canonical_transcript:ENST00000301843.13
chromosome:11
strand:1
start:70397704
end:70436584


## Étape 4 - Obtenir la séquence du gène CTTN ##

Miantenant, on veut mettre la vrai séquence du gène CTTN ;-) Encore une fois, le service REST est là pour nous aider. En regardant la doc, on voit qu'un URL contenant la commande `sequence` est utilisable pour récupérer n'importe quelle séquence (génomique ou celles des transcrits). On a simplement besoin de la valeur que nous avons mis dans le champ `id` car c'est celui qui ne change jamais pour faire notre recherche.

In [22]:
query = f"sequence/id/{aSeq.id}?type=genomic"
#
# Remarquez qu'on doit changer le format de la sortie récupérée pour
# avoir uniquement la séquence
#
reponse = requests.get(server + query, headers={ "Content-Type" : "text/plain"})

if reponse.ok:
    #
    # On met la nouvelle séquence à la place de celle du début
    #
    aSeq.seq = Seq(reponse.text)
#
# On a déjà un objet SeqRecord qui commence à avoir de l'allure
#
print(aSeq)

ID: ENSG00000085733
Name: CTTN
Description: cortactin [Source:HGNC Symbol;Acc:HGNC:3338]
Number of features: 0
/species=homo_sapiens
/assembly_name=GRCh38
/symbol=CTTN
/biotype=protein_coding
/canonical_transcript=ENST00000301843.13
/chromosome=11
/strand=1
/start=70397704
/end=70436584
Seq('AGAATAAAGACCGAGGTCTTCGCTGTGCTCTATCGGCCTCTGCATGACCTGGCC...TGA')


## Étape 5 - Créer des objets SeqFeature sur un objet SeqRecord ##

Rappel: un objet SeqFeature contient de l'information, n'importe quelle information, sur une fraction de la séquence contenue dans l'objet SeqRecord. Quel genres d'informations? Sur une protéine, ça serait par exemple des motifs de modifications post-traductionnelles; sur une séquence génomique, ça serait par exemple, l'ensemble des transcrits contenus dans Ensembl pour notre gène.

Comment faire notre requête? En regardant bien la doc,on constate qu'une simple modification de la requete `lookup` nous permet de faire ça ;-)

In [24]:
#
# En utilisant le paramètre expand avec la valeur 1 
# pour TRUE, on recevra alors tous les infos attachées
# au gène: transcrits avec leurs exons respectifs ainsi que 
# les protéines résultantes de ces transcrits
#
query = f"lookup/id/{aSeq.id}?expand=1"

reponse = requests.get(server + query, headers={ "Content-Type" : "application/json"})

if reponse.ok:
    data = reponse.json()
    #
    # Ça permet d'afficher à l'écran de manière plus
    # lisible
    #
    pprint.pprint(data)

{'Transcript': [{'Exon': [{'assembly_name': 'GRCh38',
                           'db_type': 'core',
                           'end': 70398614,
                           'id': 'ENSE00001833293',
                           'object_type': 'Exon',
                           'seq_region_name': '11',
                           'species': 'homo_sapiens',
                           'start': 70398404,
                           'strand': 1,
                           'version': 1},
                          {'assembly_name': 'GRCh38',
                           'db_type': 'core',
                           'end': 70405361,
                           'id': 'ENSE00001119698',
                           'object_type': 'Exon',
                           'seq_region_name': '11',
                           'species': 'homo_sapiens',
                           'start': 70405265,
                           'strand': 1,
                           'version': 2},
                          {'assembly_nam

## Étape 6 - Filtrer les SeqFeatures souhaités  ##

On le voit, c'est beaucoup de données... On va pas ajouter tout ça, on va simplement se concentrer sur ce qu'Ensembl appelle le [https://jun2026.archive.ensembl.org/info/genome/genebuild/canonical.html?redirect=no](transcrit canonique). Ce transcrit est considéré comme étant le plus représentatif pour un gène donné afin d'en faire l'affichage mais ça ne veut pas dire que c'est le plus biologiquement significatif pour votre recherche... Ici, on l'utilise simplement poir abréger la construction de notre objet `SeqRecord`. 

In [29]:
#
# Ici, on utilise la valeur booléenne pour TRUE (1)
# afin de ne garder que le transcrit canonique à introduire
# comme SeqFeature dans aSeq 
#
# Le champs Transcript est une liste de transcrits avec
# les infos spécifiques à chacun, y compris les exons avec
# leurs positions ainsi que l'ID de la protéine correspondante.
#
for aTranscrit in data["Transcript"]:
    if aTranscrit["is_canonical"] == 1:
        print(aTranscrit["id"])
        pprint.pprint(aTranscrit)
        for anExon in aTranscrit["Exon"]:
          exon_location = SimpleLocation(anExon["start"]-1, anExon["end"])
          exon = SeqFeature(location=exon_location,type=anExon["object_type"],qualifiers={"id": anExon["id"]})
          aSeq.features.append(exon)

for i in aSeq.features:
    print(i)



            

ENST00000301843
{'Exon': [{'assembly_name': 'GRCh38',
           'db_type': 'core',
           'end': 70398614,
           'id': 'ENSE00001330492',
           'object_type': 'Exon',
           'seq_region_name': '11',
           'species': 'homo_sapiens',
           'start': 70398529,
           'strand': 1,
           'version': 5},
          {'assembly_name': 'GRCh38',
           'db_type': 'core',
           'end': 70405361,
           'id': 'ENSE00001119698',
           'object_type': 'Exon',
           'seq_region_name': '11',
           'species': 'homo_sapiens',
           'start': 70405265,
           'strand': 1,
           'version': 2},
          {'assembly_name': 'GRCh38',
           'db_type': 'core',
           'end': 70407384,
           'id': 'ENSE00003684603',
           'object_type': 'Exon',
           'seq_region_name': '11',
           'species': 'homo_sapiens',
           'start': 70407298,
           'strand': 1,
           'version': 1},
          {'assembly_nam